# AgriEye baseline reproduction — Colab launcher

Extract `agrieye_baseline_setup.zip` into Google Drive so that `agrieye_reproduction/run.py` exists there. Put the extracted AgriEye dataset in a separate Drive folder. Choose a GPU runtime in Colab, then run cells in order.

This notebook calls the same project scripts as the local setup. It does not add the proposed illumination transforms. The documented recipe is 50 epochs; the reported final five-class model is a distinct, incompletely specified 100-epoch run. The current four-class export supports a recipe pilot, not an exact reproduction of that final model.

The project folder is on Drive to preserve checkpoints and records across runtime resets. Dataset preparation creates another working copy, so allow space for it.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/agrieye_reproduction')
DATASET_DIR = Path('/content/drive/MyDrive/AgriEye_export')  # EDIT this path
DATASET_NAME = 'agrieye_current'

assert (PROJECT_DIR / 'run.py').is_file(), 'Extract the project ZIP into Drive and set PROJECT_DIR.'
assert (DATASET_DIR / 'data.yaml').is_file(), 'Set DATASET_DIR to the extracted export containing data.yaml.'
def run(*args):
    subprocess.run([sys.executable, str(PROJECT_DIR / 'run.py'), *map(str, args)], cwd=PROJECT_DIR, check=True)

## Install and inspect the runtime

This pins Ultralytics 8.4.41 and retains a compatible Colab PyTorch installation when already present. It is not the authors' recovered environment. If a package upgrade requires a runtime restart, restart and rerun the setup cell. The environment is recorded by the doctor command and every training run.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)
run('doctor')

## Audit and prepare the working copy

Review the printed table and audit reports. This preserves supplied splits and class IDs; it does not resolve possible split leakage. A previously prepared snapshot is reused rather than overwritten.

In [ ]:
prepared = PROJECT_DIR / 'data' / 'prepared' / DATASET_NAME
if not prepared.exists():
    run('prepare', '--source', DATASET_DIR, '--name', DATASET_NAME)
else:
    print('Existing snapshot:', prepared)
    print('Training will verify its image and annotation hashes.')

## One-epoch software check

Uses at most 32 train and 16 validation images with batch 1. No test images are evaluated. Do not interpret its scores as thesis results.

In [ ]:
run('smoke', '--dataset', DATASET_NAME)

## Baseline training

Run this cell after reviewing the audit and smoke check. It uses the appendix's 50-epoch recipe with batch 16, image size 640, and early stopping patience 20. It automatically evaluates validation, not test. Keep the printed run ID.

If the GPU cannot fit batch 16, use a larger GPU for a closer recipe reproduction. An explicit smaller batch is a documented adaptation; do not silently call it the original setting. The final five-class/100-epoch candidate profile requires its matching dataset.

In [ ]:
run('train', '--dataset', DATASET_NAME,
    '--recipe', PROJECT_DIR / 'configs' / 'disease_paper_50e.yaml', '--seed', '0')

## Result tables and records

Paper values are listed separately because their exact metric namespace, split, and dataset correspondence remain unresolved. Numerical closeness alone does not establish reproduction.

In [ ]:
run('benchmark')
run('bundle-records')

## Explicit final test evaluation — only after fixing the protocol

Enter a completed baseline run ID. Do not repeatedly tune against this test set to chase the paper's score. Test access is logged.

In [ ]:
RUN_ID = ''  # Paste the exact completed baseline run ID when ready for final evaluation.
if RUN_ID:
    run('evaluate', '--run', RUN_ID, '--split', 'test')
    run('benchmark')
    run('bundle-records')
else:
    print('No test evaluation requested. Validation results remain available in runs/.')